In [ ]:
import sys
from ase import Atoms
import numpy as np
from pathlib import Path
import os
from ase.build import bulk
from ase.optimize import BFGS
from ase.visualize import view
from mace.calculators import mace_mp
from ase.build import graphene
from ase.build import bulk
from ase.calculators.emt import EMT
import json
print("PYTHON:", sys.executable)

# get rid of the future warning
import warnings 
warnings.filterwarnings(
    "ignore",
    message=r"You are using `torch.load` with `weights_only=False`.*",
    category=FutureWarning,
)



c:\Users\rnpla\anaconda3\envs\mace_phonons\lib\site-packages\e3nn\o3\_wigner.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _Jd, _W3j_flat, _W3j_indices = torch.load(o

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
PYTHON: c:\Users\rnpla\anaconda3\envs\mace_phonons\python.exe


In [ ]:
# alternatively, load the unit cell from materials projet:
# the BN.poscar was found here: 
# https://next-gen.materialsproject.org/materials/mp-984?formula=BN
# the materials project defaults to exporting .poscar to be read by vasp 


#from phonopy.interface.calculator import read_crystal_structure
#from phonopy.structure.atoms import PhonopyAtoms
#unitcell, _ = read_crystal_structure("input_files/BN.poscar", interface_mode='vasp')


# BUILDING HEXAGONAL BORON NITRIDE WITH A CARBON DIMER 
def monolayer_hbn_unitcell():
    """
    returns atoms object of the unit cell for hbn, but with a vacuum above and below
    to break the effects of potential periodicity
    """
    a = 2.504 # lattice constant  (Angstrom) (Prasad et al., 2024)
    vac = 20 # vacuum, so that even in PBC the monolayers are isolated

    # Lattice vectors, same for graphene 
    a1 = [a, 0, 0]
    a2 = [-a/2, a * np.sqrt(3)/2, 0]
    a3 = [0, 0, vac] 
    
    # use scaled positions

    hbn = Atoms(symbols=['B', 'N'],
                scaled_positions=[(0, 0, 0), # scaled positions are a proportion of the unit cell length
                                  (1/3, 2/3, 0)],
                cell=[a1, a2, a3], 
                pbc=[True, True, True])
    
    # Center the atoms in the middle of the vacuum gap
    hbn.center()
    return hbn

def bulk_hbn_unitcell(): 
    """
    returns atoms object of the unit cell for hbn. This is for bulk, so there are 
    4 atoms in the unit cell, due to the AA' spacing.

    """
    a = 2.504 # lattice constant  (Angstrom) (Prasad et al., 2024)

    c = 2*3.35  # c lattice parameter is twice the interlayer spacing in bulk h-BN (Angstrom) (Prasad et al., 2024)

    # Lattice vectors, same for graphene 
    a1 = [a, 0, 0]
    a2 = [-a/2, a * np.sqrt(3)/2, 0]
    a3 = [0, 0, c] 
    
    # use scaled positions. The same reference says hexagonal with AA' layer structure.

    scaled_pos = [
            [1/3, 2/3, 1/4], # B1
            [2/3, 1/3, 3/4], # B2
            [1/3, 2/3, 3/4], # N1
            [2/3, 1/3, 1/4]  # N2
        ]

    hbn = Atoms(symbols=['B', 'B', 'N', 'N'],
                scaled_positions=scaled_pos,
                cell=[a1, a2, a3], 
                pbc=[True, True, True])
    
    # Center the atoms in the middle of the vacuum gap
    hbn.center()
    return hbn

def diamond_unitcell():
    diamond = bulk('C', 'diamond', a=3.567)
    return diamond

def place_cc_dimer_hbn(atoms):
    
    frac = atoms.get_scaled_positions(wrap=True)
    symbols = np.array(atoms.get_chemical_symbols())
    
    # Find the Boron closest to the fractional center (0.5, 0.5, 0.5)
    b_inds = np.where(symbols == 'B')[0]
    dm = frac[b_inds] - 0.5
    dm -= np.round(dm) # Minimum Image Convention 
    iB = b_inds[np.argmin(np.linalg.norm(dm, axis=1))]
    
    # 3. Find the closest Nitrogen in the same plane
    n_inds = np.where(symbols == 'N')[0]
    # Filter for same z-layer
    same_layer = n_inds[np.isclose(frac[n_inds, 2], frac[iB, 2], atol=1e-4)]
    
    # Find bond partner using MIC distances in Cartesian space
    d_bn = frac[same_layer] - frac[iB]
    d_bn -= np.round(d_bn)
    d_cart = d_bn @ atoms.cell
    iN = same_layer[np.argmin(np.linalg.norm(d_cart, axis=1))]
    
    # 4. Swap to Carbon
    atoms[iB].symbol = 'C'
    atoms[iN].symbol = 'C'
    
    print(f"hBN Dimer: Replaced B[{iB}] and N[{iN}]")
    return atoms

def place_nv_diamond(atoms):
    # 1. Grab scaled positions and symbols
    frac = atoms.get_scaled_positions(wrap=True)
    symbols = np.array(atoms.get_chemical_symbols())
    c_inds = np.where(symbols == 'C')[0]
    
    # 2. Identify the Carbon closest to the center to be our Nitrogen
    dm = frac[c_inds] - 0.5
    dm -= np.round(dm)
    n_idx = c_inds[np.argmin(np.linalg.norm(dm, axis=1))]
    
    # 3. Find its nearest neighbor to become the Vacancy
    # get_distances is much cleaner than manual norm for bulk
    dists = atoms.get_distances(n_idx, c_inds, mic=True)
    
    # Ignore the atom itself (distance 0) to find the neighbor
    dists[c_inds == n_idx] = np.inf
    v_idx = c_inds[np.argmin(dists)]
    
    # 4. Apply changes
    atoms[n_idx].symbol = 'N'
    # Use pop for the vacancy so the index shifts are handled by ASE
    atoms.pop(v_idx)
    
    print(f"NV Center: N placed at {n_idx}, Vacancy created at {v_idx}")
    return atoms


# parameters ————————————
# supercell size: 
size = (1,1,1)

hbn = monolayer_hbn_unitcell() * size
hbn_defect = place_cc_dimer_hbn(hbn)
print(len(hbn_defect))
view(hbn_defect)

diamond = diamond_unitcell() * size
diamond_defect = place_nv_diamond(diamond)
print(len(diamond_defect))
view(diamond)





NameError: name 'np' is not defined

In [6]:
from ase.build import bulk
from ase.calculators.emt import EMT
from ase.phonons import Phonons

# Setup crystal and EMT calculator
atoms = diamond_unitcell()

# Phonon calculator
N = 7
ph = Phonons(atoms, EMT(), supercell=(N, N, N), delta=0.05)
ph.run()

# Read forces and assemble the dynamical matrix
ph.read(acoustic=True)
ph.clean()

path = atoms.cell.bandpath('FXWLΓK', npoints=100)
bs = ph.get_band_structure(path)

dos = ph.get_dos(kpts=(20, 20, 20)).sample_grid(npts=100, width=1e-3)

# Plot the band structure and DOS:
import matplotlib.pyplot as plt  # noqa

fig = plt.figure(figsize=(7, 4))
ax = fig.add_axes([0.12, 0.07, 0.67, 0.85])

emax = 0.035
bs.plot(ax=ax, emin=0.0, emax=emax)

dosax = fig.add_axes([0.8, 0.07, 0.17, 0.85])
dosax.fill_between(
    dos.get_weights(),
    dos.get_energies(),
    y2=0,
    color='grey',
    edgecolor='k',
    lw=1,
)

dosax.set_ylim(0, emax)
dosax.set_yticks([])
dosax.set_xticks([])
dosax.set_xlabel('DOS', fontsize=18)

fig.savefig('Al_phonon.png')

KeyError: 'F'

In [ ]:
# We have the structure. Now we want to get this working in phonopy. 
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms

# the BN.poscar was found here: 
# https://next-gen.materialsproject.org/materials/mp-984?formula=BN

def ase_to_phonopy_atoms(atoms):
    """ASE Atoms -> PhonopyAtoms."""
    return PhonopyAtoms(
        symbols=atoms.get_chemical_symbols(),
        cell=atoms.cell.array,
        scaled_positions=atoms.get_scaled_positions(),
    )

model = "small-omat-0"
device = "cuda"
dtype = "float64"  # important for phonons

Using Materials Project MACE for MACECalculator with C:\Users\rnpla\.cache\mace/20231203mace128L1_epoch199model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
Using head Default out of ['Default']
Default dtype float32 does not match model dtype float64, converting models to float32.
WARNING, 3 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 1.704e-04*i)
WARNING, 3 imaginary frequencies at q = ( 0.02,  0.00,  0.02) ; (omega_q = 1.704e-04*i)
WARNING, 3 imaginary frequencies at q = ( 0.04,  0.00,  0.04) ; (omega_q = 1.704e-04*i)
WARNING, 3 imaginary frequencies at q = ( 0.06,  0.00,  0.06) ; (omega_q = 1.704e-04*i)
WARNING, 3 imaginary frequencies at q = ( 0.08,  0.00,  0.08) ; (omega_q = 1.704e-04*i)
WARNING, 3 imaginary frequencies at q = ( 0.10,  0.00,  0.10) ; (omega_q = 1.704e-04*i)
WARNING, 3 imaginary frequencies at q = ( 0.12,  0.00,  0.12) ; (omega_q = 1.704e-04*i)
WARNING, 3 imagina